In [9]:
import pandas as pd
import random
import math
import os

# Saturasyon büyüme parametreleri
yayilma_parametreleri = {
    0: (0.05, 0.1),
    1: (0.1, 0.125),
    2: (0.125, 0.167),
    3: (0.167, 0.25),
    4: (0.25, 0.5),
    5: (0.5, 1)
}

# Yarıçap artışı (metre)
mesafe_artis = {
    0: 50,
    1: 100,
    2: 150,
    3: 200,
    4: 250,
    5: 300
}

# Haversine mesafe hesaplama fonksiyonu
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # radius of Earth in meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Dosya yolları
base_path = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı"
kullanici_girdisi_path = os.path.join(base_path, "kullanici_girdisi", "kullanici_girdisi4.xlsx")
bina_alan_verisi_path = os.path.join(base_path, "construction_areas2.xlsx")
ana_algoritma_path = r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\new_buildings_2024_2035_1703.xlsx'  # Ana algoritma sonuç dosyası

# Kullanıcı girdisi, bölge verisi ve bina alanı verilerini oku
kullanici_girdisi = pd.read_excel(kullanici_girdisi_path, sheet_name="kullanici_girdisi")
bolge_verisi = pd.read_excel(kullanici_girdisi_path, sheet_name="bolge")
bina_alan_verisi = pd.read_excel(bina_alan_verisi_path, sheet_name='areas')

# Bina alanı verisini sözlüğe çevir
bina_alanlari = {}
for _, row in bina_alan_verisi.iterrows():
    bina_alanlari[row['imar_tipi']] = row['bina_başı_brüt_alan']

# Saturasyon hızını statik bir değerden al
saturation_speed = kullanici_girdisi["saturation_speed"].iloc[0]

# Hücre merkezlerini hesapla
bolge_verisi["center_lat"] = (bolge_verisi["top"] + bolge_verisi["bottom"]) / 2
bolge_verisi["center_lon"] = (bolge_verisi["left"] + bolge_verisi["right"]) / 2

# Çıktı dosya yolları
imar_output_path = os.path.join(base_path, "kullanici_girdisi", "imar_oranlari_horizon7_04_6.xlsx")
saturasyon_output_path = os.path.join(base_path, "kullanici_girdisi", "saturasyon_horizon07_04_6.xlsx")
bina_sayisi_output_path = os.path.join(base_path, "kullanici_girdisi", "bina_sayilari_horizon07_04_6.xlsx")
birlestirilmis_output_path = os.path.join(base_path, "kullanici_girdisi", "birlestirilmis_sonuc2.xlsx")

# Excel writer'lar
imar_writer = pd.ExcelWriter(imar_output_path, engine="xlsxwriter")
saturasyon_writer = pd.ExcelWriter(saturasyon_output_path, engine="xlsxwriter")
bina_sayisi_writer = pd.ExcelWriter(bina_sayisi_output_path, engine="xlsxwriter")
birlestirilmis_writer = pd.ExcelWriter(birlestirilmis_output_path, engine="xlsxwriter")

# Başlangıç yılı ve horizon
start_year = kullanici_girdisi["start_year"].iloc[0]
min_year = 2024  # Minimum yıl
end_year = 2035

# Alt kırılım kolonlarını tespit et (1-2_KATLI_MESKEN gibi imar tiplerini içeren kolonlar)
imar_kolonlari = [col for col in kullanici_girdisi.columns if col not in ['id', 'start_year', 'saturation_speed']]

# Tüm hücrelerde saturasyon başlangıç değeri
tum_hucreler = bolge_verisi.copy()
tum_hucreler["saturasyon"] = 0

# Başlangıç için imar tiplerini 0 olarak başlat
for kolon in imar_kolonlari:
    tum_hucreler[kolon] = 0

# Ana algoritma sonuçlarını oku - tüm yılları içeren bir sözlük oluştur
ana_algoritma_sonuclari = {}
try:
    # Excel dosyasındaki tüm sayfaları oku
    excel_file = pd.ExcelFile(ana_algoritma_path)
    
    # Tüm yıllar için verileri oku ve sözlüğe ekle
    for sheet_name in excel_file.sheet_names:
        try:
            year = int(sheet_name)  # Sayfa adı yıl olmalı
            ana_algoritma_sonuclari[year] = pd.read_excel(ana_algoritma_path, sheet_name=sheet_name)
        except ValueError:
            # Sayfa adı yıl değilse atla
            continue
except Exception as e:
    print(f"Ana algoritma sonuçları okunamadı: {str(e)}")
    # Ana algoritma dosyası yoksa boş bir sözlük kullan
    ana_algoritma_sonuclari = {}

# Start year'dan önceki yılları birleştirilmiş tabloya direk kopyala
for year in range(min_year, start_year):
    if year in ana_algoritma_sonuclari:
        ana_algoritma_sonuclari[year].to_excel(birlestirilmis_writer, sheet_name=str(year), index=False)
        print(f"{year} yılı verileri doğrudan kopyalandı.")

# Önceki yılın bina sayılarını tutacak değişken
onceki_yil_bina_sayilari = None

# Başlangıç yılı için hesaplama
merkez_hucreler = []
for index, row in kullanici_girdisi.iterrows():
    id_ = row['id']
    merkez_hucreler.append(id_)
    
    # Saturasyon büyümesini rastgele belirleme
    saturation_range = yayilma_parametreleri[saturation_speed]
    saturation_growth = random.uniform(*saturation_range)
    tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"] += saturation_growth
    tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"] = tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"].clip(upper=1)
    
    # Kullanıcı girdisinden direkt alt kırılım değerlerini al
    for kolon in imar_kolonlari:
        tum_hucreler.loc[tum_hucreler['id'] == id_, kolon] = row[kolon]

# Merkez konumu sabitle
merkez_lat = tum_hucreler[tum_hucreler['id'].isin(merkez_hucreler)]["center_lat"].mean()
merkez_lon = tum_hucreler[tum_hucreler['id'].isin(merkez_hucreler)]["center_lon"].mean()

# Başlangıç yılı için bina sayılarını hesapla
bina_sayilari = tum_hucreler[["id"]].copy()
for kolon in imar_kolonlari:
    # İnşa edilebilir alan = hücre alanı * saturasyon * imar oranı
    insa_edilebilir_alan = tum_hucreler["cell_area"] * tum_hucreler["saturasyon"] * tum_hucreler[kolon]
    # Bina sayısı = inşa edilebilir alan / bina başı brüt alan
    bina_sayilari[kolon] = insa_edilebilir_alan / bina_alanlari[kolon]
    # Virgülden sonra 2 basamağa yuvarla
    bina_sayilari[kolon] = bina_sayilari[kolon].round(2)

# İlk yıl için yeni eklenen binalar (tümü yeni)
yeni_eklenen_binalar = bina_sayilari.copy()

# Ana algoritma sonuçlarını güncelle - ilk yıl
if start_year in ana_algoritma_sonuclari:
    # Ana algoritma sonuçlarını al
    ana_sonuc = ana_algoritma_sonuclari[start_year].copy()
    
    # Yeni eklenen binaları ana sonuç tablosuna ekle
    birlestirilmis_sonuc = pd.merge(
        ana_sonuc,
        yeni_eklenen_binalar,
        on='id',
        how='left',
        suffixes=('', '_yeni')
    )
    
    # Yeni eklenen sütunları temizle ve ana sütunları güncelle
    for kolon in imar_kolonlari:
        if f'{kolon}_yeni' in birlestirilmis_sonuc.columns:
            # Eksik değerleri 0 ile doldur
            birlestirilmis_sonuc[f'{kolon}_yeni'] = birlestirilmis_sonuc[f'{kolon}_yeni'].fillna(0)
            
            # Ana algoritma sonucunda kolon varsa güncelle, yoksa ekle
            if kolon in birlestirilmis_sonuc.columns:
                birlestirilmis_sonuc[kolon] = birlestirilmis_sonuc[kolon] + birlestirilmis_sonuc[f'{kolon}_yeni']
            else:
                birlestirilmis_sonuc[kolon] = birlestirilmis_sonuc[f'{kolon}_yeni']
            
            # Geçici sütunu sil
            birlestirilmis_sonuc = birlestirilmis_sonuc.drop(columns=[f'{kolon}_yeni'])
    
    # Birleştirilmiş sonucu Excel'e yaz
    birlestirilmis_sonuc.to_excel(birlestirilmis_writer, sheet_name=str(start_year), index=False)
else:
    # Ana algoritma sonucu yoksa sadece yeni eklenen binaları yaz
    yeni_eklenen_binalar.to_excel(birlestirilmis_writer, sheet_name=str(start_year), index=False)

# İlk yılın sonuçlarını yaz
imar_sonuc = tum_hucreler[["id"] + imar_kolonlari]
imar_sonuc.to_excel(imar_writer, sheet_name=str(start_year), index=False)
saturasyon_sonuc = tum_hucreler[["id", "saturasyon"]]
saturasyon_sonuc.to_excel(saturasyon_writer, sheet_name=str(start_year), index=False)
bina_sayilari.to_excel(bina_sayisi_writer, sheet_name=str(start_year), index=False)

# Önceki yıl bina sayılarını güncelle
onceki_yil_bina_sayilari = bina_sayilari.copy()

# İteratif yıllık işlem
for year in range(start_year + 1, end_year + 1):
    yeni_hucreler = []

    # Mevcut saturasyon seviyesini artır
    for index, row in tum_hucreler.iterrows():
        if row["saturasyon"] > 0 and row["saturasyon"] < 1:
            saturation_range = yayilma_parametreleri[saturation_speed]
            saturation_growth = random.uniform(*saturation_range)
            if row['id'] in kullanici_girdisi['id'].values:
                tum_hucreler.loc[index, "saturasyon"] += saturation_growth
            else:
                tum_hucreler.loc[index, "saturasyon"] += saturation_growth / 2
            tum_hucreler.loc[index, "saturasyon"] = min(tum_hucreler.loc[index, "saturasyon"], 1)

    # Henüz saturasyon başlamamış hücrelere yayılma
    for index, row in tum_hucreler.iterrows():
        if row["saturasyon"] == 0:            
            mesafe = haversine(merkez_lat, merkez_lon, row["center_lat"], row["center_lon"])
            if mesafe <= mesafe_artis[saturation_speed] * (year - start_year):
                saturation_range = yayilma_parametreleri[saturation_speed]
                saturation_growth = random.uniform(*saturation_range) / 2
                tum_hucreler.loc[index, "saturasyon"] += saturation_growth
                tum_hucreler.loc[index, "saturasyon"] = min(tum_hucreler.loc[index, "saturasyon"], 1)
                yeni_hucreler.append(index)

    # Yeni merkez hücreler seti güncelle
    merkez_hucreler = yeni_hucreler

    # İmar oranlarını hesapla - sadece değeri 0'dan büyük olan hücrelerin ortalamasını al
    for idx in yeni_hucreler:
        komsu_hucreler = tum_hucreler.apply(
            lambda row: haversine(tum_hucreler.loc[idx, "center_lat"], tum_hucreler.loc[idx, "center_lon"], row["center_lat"], row["center_lon"]) <= 300
            and row["saturasyon"] > 0, axis=1)
        komsu_hucreler = tum_hucreler[komsu_hucreler]
        for kolon in imar_kolonlari:
            if not komsu_hucreler.empty:
                # Sadece 0'dan büyük değerlerin ortalamasını al
                pozitif_degerler = komsu_hucreler[komsu_hucreler[kolon] > 0][kolon]
                if not pozitif_degerler.empty:
                    tum_hucreler.loc[idx, kolon] = pozitif_degerler.mean()
                else:
                    tum_hucreler.loc[idx, kolon] = 0

    # Bina sayılarını hesapla
    bina_sayilari = tum_hucreler[["id"]].copy()
    for kolon in imar_kolonlari:
        # İnşa edilebilir alan = hücre alanı * saturasyon * imar oranı
        insa_edilebilir_alan = tum_hucreler["cell_area"] * tum_hucreler["saturasyon"] * tum_hucreler[kolon]
        # Bina sayısı = inşa edilebilir alan / bina başı brüt alan
        bina_sayilari[kolon] = insa_edilebilir_alan / bina_alanlari[kolon]
        # Virgülden sonra 2 basamağa yuvarla
        bina_sayilari[kolon] = bina_sayilari[kolon].round(2)

    # Yeni eklenen binaları hesapla
    yeni_eklenen_binalar = bina_sayilari.copy()
    # Tüm imar tipleri için artışı hesapla
    for kolon in imar_kolonlari:
        # İki DataFrame'i id'ye göre birleştir
        merged = pd.merge(bina_sayilari[['id', kolon]], 
                         onceki_yil_bina_sayilari[['id', kolon]], 
                         on='id', 
                         how='left', 
                         suffixes=('_yeni', '_eski'))
        # Eksik değerleri 0 ile doldur
        merged = merged.fillna(0)
        # Artış miktarını hesapla (negatif artış olamaz)
        merged['artis'] = merged.apply(lambda row: max(row[f'{kolon}_yeni'] - row[f'{kolon}_eski'], 0), axis=1)
        # Yeni eklenen binalar tablosunu güncelle
        yeni_eklenen_binalar.loc[yeni_eklenen_binalar['id'].isin(merged['id']), kolon] = merged['artis'].values

    # Ana algoritma sonuçlarını güncelle
    if year in ana_algoritma_sonuclari:
        # Ana algoritma sonuçlarını al
        ana_sonuc = ana_algoritma_sonuclari[year].copy()
        
        # Yeni eklenen binaları ana sonuç tablosuna ekle
        birlestirilmis_sonuc = pd.merge(
            ana_sonuc,
            yeni_eklenen_binalar,
            on='id',
            how='left',
            suffixes=('', '_yeni')
        )
        
        # Yeni eklenen sütunları temizle ve ana sütunları güncelle
        for kolon in imar_kolonlari:
            if f'{kolon}_yeni' in birlestirilmis_sonuc.columns:
                # Eksik değerleri 0 ile doldur
                birlestirilmis_sonuc[f'{kolon}_yeni'] = birlestirilmis_sonuc[f'{kolon}_yeni'].fillna(0)
                
                # Ana algoritma sonucunda kolon varsa güncelle, yoksa ekle
                if kolon in birlestirilmis_sonuc.columns:
                    birlestirilmis_sonuc[kolon] = birlestirilmis_sonuc[kolon] + birlestirilmis_sonuc[f'{kolon}_yeni']
                else:
                    birlestirilmis_sonuc[kolon] = birlestirilmis_sonuc[f'{kolon}_yeni']
                
                # Geçici sütunu sil
                birlestirilmis_sonuc = birlestirilmis_sonuc.drop(columns=[f'{kolon}_yeni'])
        
        # Birleştirilmiş sonucu Excel'e yaz
        birlestirilmis_sonuc.to_excel(birlestirilmis_writer, sheet_name=str(year), index=False)
    else:
        # Ana algoritma sonucu yoksa sadece yeni eklenen binaları yaz
        yeni_eklenen_binalar.to_excel(birlestirilmis_writer, sheet_name=str(year), index=False)

    # Sonuçları yaz
    imar_sonuc = tum_hucreler[["id"] + imar_kolonlari]
    imar_sonuc.to_excel(imar_writer, sheet_name=str(year), index=False)
    saturasyon_sonuc = tum_hucreler[["id", "saturasyon"]]
    saturasyon_sonuc.to_excel(saturasyon_writer, sheet_name=str(year), index=False)
    bina_sayilari.to_excel(bina_sayisi_writer, sheet_name=str(year), index=False)
    
    # Önceki yıl bina sayılarını güncelle
    onceki_yil_bina_sayilari = bina_sayilari.copy()

# Dosyaları kaydet
imar_writer.close()
saturasyon_writer.close()
bina_sayisi_writer.close()
birlestirilmis_writer.close()

print("Tüm horizon tamamlandı ve sonuçlar kaydedildi. Birleştirilmiş sonuçlar da oluşturuldu.")

2024 yılı verileri doğrudan kopyalandı.
2025 yılı verileri doğrudan kopyalandı.
2026 yılı verileri doğrudan kopyalandı.
2027 yılı verileri doğrudan kopyalandı.


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_21492\1625420583.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.22241167]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"] += saturation_growth
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_21492\1625420583.py:129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.053667948317455486' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  tum_hucreler.loc[tum_hucreler['id'] == id_, kolon] = row[kolon]
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_21492\1625420583.py:129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.1380938916194824' has dtype incompa

Tüm horizon tamamlandı ve sonuçlar kaydedildi. Birleştirilmiş sonuçlar da oluşturuldu.
